# Transformacion y Clasificacion KNN con Excel

**Autor:** Daniel Antonio Guanga Gallardo  
**Carrera:** Computacion  
**Materia:** Inteligencia Artificial

Este notebook muestra, paso a paso, como construir un flujo completo de Machine Learning usando un archivo Excel pequeno (`dataIA.xlsx`).

## Objetivo
Predecir el nivel de satisfaccion (`nivel_satisfaccion`) a partir de variables de entrada (`sexo`, `edad`, `pais`) usando un modelo KNN (K-Nearest Neighbors).

## Que aprenderas en este notebook
1. Como limpiar y estandarizar texto y nombres de columnas.
2. Como transformar variables categoricas y numericas con `ColumnTransformer`.
3. Como entrenar y evaluar un modelo con validacion Leave-One-Out.
4. Como guardar el modelo y exportar los datos transformados.
5. Como hacer predicciones sobre nuevos registros.

## 1. Importaciones y rutas

En esta seccion se cargan las librerias necesarias y se definen rutas de trabajo.

- `pandas` y `numpy`: manejo y transformacion de datos.
- `scikit-learn`: preprocesamiento, entrenamiento y evaluacion del modelo.
- `pickle`: guardado del modelo entrenado.
- `Path`: construccion de rutas de forma portable.

Tambien se crean diccionarios para mapear etiquetas de texto a clases numericas (y viceversa).

In [1]:
from __future__ import annotations

import pickle
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import LeaveOneOut, cross_val_predict, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

BASE_DIR = Path.cwd()
DATA_PATH = BASE_DIR / 'dataIA.xlsx'
MODEL_PATH = BASE_DIR / 'modelo_knn_satisfaccion.pkl'
TRANSFORMED_PATH = BASE_DIR / 'dataset_transformado_satisfaccion.csv'

TARGET_MAP = {
    'no me gusta': 0,
    'neutral': 1,
    'neutro': 1,
    'me gusta': 2,
}

TARGET_LABELS = {
    0: 'no me gusta',
    1: 'neutral',
    2: 'me gusta',
}

### Explicacion del bloque anterior (importaciones y configuracion)

1. Se importan modulos de Python estandar (`pickle`, `unicodedata`, `Path`) y librerias de ciencia de datos.
2. Se definen rutas para leer el Excel, guardar el modelo y exportar el CSV transformado.
3. `TARGET_MAP` convierte textos de satisfaccion a clases numericas.
4. `TARGET_LABELS` permite volver de numero a texto para explicar resultados de prediccion.

## 2. Funciones auxiliares

Esta seccion encapsula la logica principal en funciones reutilizables.

- `clean_text`: normaliza texto para evitar problemas de acentos y mayusculas.
- `clean_column_name`: estandariza nombres de columnas.
- `make_one_hot_encoder`: crea `OneHotEncoder` compatible con distintas versiones de scikit-learn.
- `load_data`: carga, limpia y valida el Excel.
- `build_pipeline`: crea el pipeline de preprocesamiento + KNN.
- `save_model` y `load_model`: guardan/cargan el modelo entrenado.
- `encode_dataset`: exporta las columnas transformadas con el mismo formato del Excel (`sexo_*`, `pais_*`, `nSatis_*`).
- `validate_transformations`: verifica media 0 y desviacion estandar 1.
- `predict_new_sample`: estima la clase de un nuevo ejemplo y su certeza.

In [2]:
def clean_text(value: object) -> str:
    text = unicodedata.normalize("NFKD", str(value))
    text = text.encode("ascii", "ignore").decode("ascii")
    return text.strip().lower()


def clean_column_name(value: object) -> str:
    text = clean_text(value)
    text = text.replace(" ", "_")
    text = text.replace("-", "_")
    return text


def make_one_hot_encoder() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def load_data(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"No se encontro el archivo Excel: {path}")

    df = pd.read_excel(path)
    df.columns = [clean_column_name(column) for column in df.columns]

    rename_map = {}
    if "pas" in df.columns and "pais" not in df.columns:
        rename_map["pas"] = "pais"
    if "nivelsatisfaccion" in df.columns:
        rename_map["nivelsatisfaccion"] = "nivel_satisfaccion"

    if rename_map:
        df = df.rename(columns=rename_map)

    required_columns = {"sexo", "edad", "pais", "nivel_satisfaccion"}
    missing = required_columns - set(df.columns)
    if missing:
        raise ValueError(f"Faltan columnas requeridas en el Excel: {sorted(missing)}")

    df = df.copy()
    df["nivel_satisfaccion"] = df["nivel_satisfaccion"].map(lambda value: TARGET_MAP.get(clean_text(value), np.nan))
    df = df.dropna(subset=["nivel_satisfaccion"])
    df["nivel_satisfaccion"] = df["nivel_satisfaccion"].astype(int)
    df["edad"] = pd.to_numeric(df["edad"], errors="coerce")
    df = df.dropna(subset=["edad"])
    df["edad"] = df["edad"].astype(float)
    return df.reset_index(drop=True)


def build_pipeline() -> Pipeline:
    categorical_features = ["sexo", "pais"]
    numeric_features = ["edad"]

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_one_hot_encoder()),
            ("scaler", StandardScaler()),
        ]
    )

    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("cat", categorical_transformer, categorical_features),
            ("num", numeric_transformer, numeric_features),
        ],
        remainder="drop",
    )

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("knn", KNeighborsClassifier(n_neighbors=3, metric="minkowski")),
        ]
    )


def save_model(model: Pipeline, path: Path) -> None:
    with path.open("wb") as handle:
        pickle.dump(model, handle, protocol=pickle.HIGHEST_PROTOCOL)


def load_model(path: Path) -> Pipeline:
    with path.open("rb") as handle:
        return pickle.load(handle)


def encode_dataset(model: Pipeline, X: pd.DataFrame, y: pd.Series) -> pd.DataFrame:
    transformed = model.named_steps["preprocessor"].transform(X)
    feature_names = model.named_steps["preprocessor"].get_feature_names_out()
    transformed_df = pd.DataFrame(transformed, columns=feature_names)

    base_df = pd.DataFrame(
        {
            "sexo_M": transformed_df["cat__sexo_F"],
            "sexo_F": transformed_df["cat__sexo_M"],
            "edad": transformed_df["num__edad"],
            "pais_Brasil": transformed_df["cat__pais_Brasil"],
            "pais_Chile": transformed_df["cat__pais_Chile"],
            "pais_Ecuador": transformed_df["cat__pais_Ecuador"],
            "pais_España": transformed_df["cat__pais_España"],
        }
    )

    target_text = y.map(TARGET_LABELS)
    target_one_hot = pd.get_dummies(target_text)

    ordered_target_cols = ["me gusta", "neutral", "no me gusta"]
    for col in ordered_target_cols:
        if col not in target_one_hot.columns:
            target_one_hot[col] = 0

    target_one_hot = target_one_hot[ordered_target_cols]
    target_scaled = pd.DataFrame(
        StandardScaler().fit_transform(target_one_hot),
        columns=["nSatis_meGusta", "nSatis_neutral", "nSatis_noMeGusta"],
        index=target_one_hot.index,
    )

    final_df = pd.concat([base_df, target_scaled], axis=1)
    return final_df


def validate_transformations(model: Pipeline, X: pd.DataFrame, y: pd.Series) -> None:
    transformed = model.named_steps["preprocessor"].transform(X)
    feature_names = model.named_steps["preprocessor"].get_feature_names_out()

    print("\nValidacion de transformaciones:")
    print("=" * 60)
    for i, feature in enumerate(feature_names):
        media = float(np.mean(transformed[:, i]))
        desv_std = float(np.std(transformed[:, i], ddof=0))
        print(f"{feature:25} | Media: {media:8.6f} | Desv.Est: {desv_std:8.6f}")

    target_scaler = StandardScaler()
    y_transformed = target_scaler.fit_transform(y.values.reshape(-1, 1)).flatten()
    media_y = float(np.mean(y_transformed))
    desv_std_y = float(np.std(y_transformed, ddof=0))
    print(f"{'nivel_satisfaccion':25} | Media: {media_y:8.6f} | Desv.Est: {desv_std_y:8.6f}")
    print("=" * 60)


def predict_new_sample(model: Pipeline, sexo: str, edad: float, pais: str) -> pd.DataFrame:
    sample = pd.DataFrame(
        [{
            "sexo": sexo,
            "edad": edad,
            "pais": pais,
        }]
    )
    pred_class = int(model.predict(sample)[0])
    proba = model.predict_proba(sample)[0]
    confidence = float(np.max(proba))
    return pd.DataFrame(
        [
            {
                "prediccion_numero": pred_class,
                "prediccion_texto": TARGET_LABELS.get(pred_class, "desconocido"),
                "certeza": round(confidence, 4),
            }
        ]
    )

### Explicacion del bloque anterior (funciones auxiliares)

- `clean_text` y `clean_column_name` evitan errores de codificacion y hacen el dataset mas consistente.
- `load_data` incluye validaciones clave: comprueba columnas esperadas y limpia tipos de dato.
- `build_pipeline` define un flujo reproducible: imputacion, codificacion, escalado y clasificacion.
- `save_model` permite persistir el entrenamiento.
- `predict_new_sample` empaqueta una prediccion completa (clase + certeza) para nuevos datos.

Esta modularidad facilita mantenimiento, pruebas y reutilizacion del codigo.

## 3. Carga y vista del dataset

Aqui se ejecuta `load_data(DATA_PATH)` para:

1. Leer el Excel.
2. Estandarizar nombres de columnas.
3. Corregir posibles problemas de codificacion en `pais`.
4. Convertir `nivel_satisfaccion` a valores numericos.
5. Limpiar filas invalidas (si existieran).

Al final se visualiza el dataset ya limpio y la distribucion de la variable objetivo.

In [3]:
df = load_data(DATA_PATH)
print('Dataset cargado:')
display(df)

print('Distribucion de la variable objetivo:')
display(df['nivel_satisfaccion'].value_counts().sort_index())

Dataset cargado:


,sexo,edad,pais,nivel_satisfaccion
0,F,65.0,Brasil,2
1,M,26.0,España,0
2,F,21.0,Chile,1
3,M,12.0,Ecuador,0
4,F,32.0,España,2
5,M,9.0,Ecuador,0


Distribucion de la variable objetivo:


nivel_satisfaccion
0    3
1    1
2    2
Name: count, dtype: int64

### Como interpretar la salida de carga

- Si el dataframe se muestra correctamente, la lectura de Excel fue exitosa.
- La distribucion de `nivel_satisfaccion` te indica si hay clases desbalanceadas.
- Si faltara una columna o hubiera valores invalidos, el notebook fallaria aqui de forma controlada (validacion temprana).

## 4. Entrenamiento y evaluacion (Leave-One-Out)

En esta etapa se separan variables de entrada (`X`) y salida (`y`), se construye el modelo y se evalua con validacion cruzada Leave-One-Out (LOO).

### Por que Leave-One-Out aqui
Como el dataset es muy pequeno, LOO permite entrenar con casi todos los datos y probar en 1 registro por iteracion. Asi se aprovecha mejor la informacion disponible.

### Metricas mostradas
- `Accuracy promedio (LOO)`: promedio de aciertos en todas las iteraciones.
- `Accuracy global`: aciertos totales entre todas las predicciones.
- `Matriz de confusion`: errores y aciertos por clase.
- `Classification report`: precision, recall y F1 por clase.

In [4]:
X = df[['sexo', 'edad', 'pais']]
y = df['nivel_satisfaccion']

model = build_pipeline()
cv = LeaveOneOut()

scores = cross_val_score(model, X, y, cv=cv)
predictions = cross_val_predict(model, X, y, cv=cv)

print('Accuracy promedio (LOO):', round(float(scores.mean()), 4))
print('Accuracy global:', round(float(accuracy_score(y, predictions)), 4))
print('Matriz de confusion:')
print(confusion_matrix(y, predictions))
print('Reporte de clasificacion:')
print(classification_report(
    y,
    predictions,
    target_names=[TARGET_LABELS[index] for index in sorted(TARGET_LABELS)],
    zero_division=0
))

Accuracy promedio (LOO): 0.5
Accuracy global: 0.5
Matriz de confusion:
[[3 0 0]
 [0 0 1]
 [2 0 0]]
Reporte de clasificacion:
              precision    recall  f1-score   support

 no me gusta       0.60      1.00      0.75         3
     neutral       0.00      0.00      0.00         1
    me gusta       0.00      0.00      0.00         2

    accuracy                           0.50         6
   macro avg       0.20      0.33      0.25         6
weighted avg       0.30      0.50      0.38         6



### Como interpretar la evaluacion

1. Un `accuracy` cercano a 1.0 indica buen rendimiento en este conjunto.
2. La matriz de confusion muestra en que clases se equivoca el modelo.
3. El reporte por clase ayuda a detectar si una etiqueta se predice peor que otras.

Nota: al tener pocos datos, las metricas pueden variar bastante y deben tomarse como orientativas.

## 5. Ajuste final, guardado y exportacion

Despues de evaluar, se entrena el modelo final con todos los datos (`model.fit(X, y)`) para dejarlo listo para uso real.

Ademas:
- Se valida la estandarizacion con `validate_transformations` (media 0, desviacion estandar 1).
- Se guarda el modelo en `modelo_knn_satisfaccion.pkl`.
- Se exporta el dataset transformado con el mismo formato de Excel usando `encode_dataset`.

Esto permite reutilizar el modelo sin volver a entrenar y mantener coherencia entre Excel y Python.

In [5]:
model.fit(X, y)
validate_transformations(model, X, y)
save_model(model, MODEL_PATH)
print("Modelo guardado en:", MODEL_PATH)

transformed_df = encode_dataset(model, X, y)
transformed_df.to_csv(TRANSFORMED_PATH, index=False, encoding="utf-8-sig")
print("Dataset transformado guardado en:", TRANSFORMED_PATH)
display(transformed_df.head())


Validacion de transformaciones:
cat__sexo_F               | Media: 0.000000 | Desv.Est: 1.000000
cat__sexo_M               | Media: 0.000000 | Desv.Est: 1.000000
cat__pais_Brasil          | Media: 0.000000 | Desv.Est: 1.000000
cat__pais_Chile           | Media: 0.000000 | Desv.Est: 1.000000
cat__pais_Ecuador         | Media: 0.000000 | Desv.Est: 1.000000
cat__pais_España          | Media: 0.000000 | Desv.Est: 1.000000
num__edad                 | Media: 0.000000 | Desv.Est: 1.000000
nivel_satisfaccion        | Media: -0.000000 | Desv.Est: 1.000000
Modelo guardado en: c:\Users\User\Downloads\IA\DataIA\modelo_knn_satisfaccion.pkl
Dataset transformado guardado en: c:\Users\User\Downloads\IA\DataIA\dataset_transformado_satisfaccion.csv


,sexo_M,sexo_F,edad,pais_Brasil,pais_Chile,pais_Ecuador,pais_España,nSatis_meGusta,nSatis_neutral,nSatis_noMeGusta
0,1.0,-1.0,2.027027,2.236068,-0.447214,-0.707107,-0.707107,1.414214,-0.447214,-1.0
1,-1.0,1.0,-0.081081,-0.447214,-0.447214,-0.707107,1.414214,-0.707107,-0.447214,1.0
2,1.0,-1.0,-0.351351,-0.447214,2.236068,-0.707107,-0.707107,-0.707107,2.236068,-1.0
3,-1.0,1.0,-0.837838,-0.447214,-0.447214,1.414214,-0.707107,-0.707107,-0.447214,1.0
4,1.0,-1.0,0.243243,-0.447214,-0.447214,-0.707107,1.414214,1.414214,-0.447214,-1.0


### Resultado del guardado y exportacion

- El archivo `.pkl` contiene el pipeline completo (preprocesamiento + modelo).
- El `.csv` transformado te deja ver exactamente que variables genera `OneHotEncoder` y como queda `edad` tras `StandardScaler`.

Esto es util para trazabilidad del modelo y para compartir insumos con otros analistas.

## 6. Prediccion de un nuevo registro

Aqui se prueba el modelo con un ejemplo manual (`sexo='F', edad=30, pais='Chile'`).

La salida incluye:
- `prediccion_numero`: clase numerica estimada.
- `prediccion_texto`: etiqueta legible de la clase.
- `certeza`: probabilidad asociada a la clase predicha (valor maximo de `predict_proba`).

Puedes cambiar estos valores para simular nuevos casos.

In [ ]:
pred_ejemplo = predict_new_sample(model, sexo='F', edad=30, pais='Chile')
display(pred_ejemplo)

### Lectura de la prediccion final

- `prediccion_numero`: salida codificada del modelo.
- `prediccion_texto`: version comprensible para usuario final.
- `certeza`: confianza del modelo en esa prediccion.

Si quieres hacer pruebas adicionales, cambia `sexo`, `edad` y `pais` en la celda anterior y vuelve a ejecutar.

## 7. Comparacion automatica: Python vs Excel

Esta seccion compara automaticamente el archivo exportado por Python (`dataset_transformado_satisfaccion.csv`) contra la tabla de estandarizacion del Excel (`1.Transformaciones Variables.xlsx`).

La validacion revisa:
- Nombres y orden de columnas.
- Numero de filas comparables.
- Diferencia numerica absoluta por columna.
- Coincidencia dentro de una tolerancia configurable.

In [8]:
import pandas as pd
import numpy as np
import unicodedata

expected_cols = [
    "sexo_M", "sexo_F", "edad", "pais_Brasil", "pais_Chile",
    "pais_Ecuador", "pais_España", "nSatis_meGusta",
    "nSatis_neutral", "nSatis_noMeGusta",
]

def norm_text(value: object) -> str:
    text = "" if pd.isna(value) else str(value)
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")
    text = text.strip().lower().replace(" ", "_")
    return text

python_df = pd.read_csv(TRANSFORMED_PATH, encoding="utf-8-sig")
python_num = python_df[expected_cols].apply(pd.to_numeric, errors="coerce")

excel_path = BASE_DIR / "1.Transformaciones Variables.xlsx"
if not excel_path.exists():
    raise FileNotFoundError(f"No se encontro el archivo esperado: {excel_path}")

target_norm = [norm_text(c) for c in expected_cols]
candidates = []

for sheet in pd.ExcelFile(excel_path).sheet_names:
    raw = pd.read_excel(excel_path, sheet_name=sheet, header=None)

    for r in range(raw.shape[0]):
        row_norm = [norm_text(v) for v in raw.iloc[r].tolist()]
        if all(col in row_norm for col in target_norm):
            col_idx = [row_norm.index(col) for col in target_norm]

            data_rows = []
            rr = r + 1
            while rr < raw.shape[0]:
                values = [raw.iat[rr, c] for c in col_idx]
                if all(pd.isna(v) or str(v).strip() == "" for v in values):
                    break
                data_rows.append(values)
                rr += 1

            if not data_rows:
                continue

            excel_block = pd.DataFrame(data_rows, columns=expected_cols)
            excel_num = excel_block.apply(pd.to_numeric, errors="coerce")

            n = min(len(python_num), len(excel_num))
            if n == 0:
                continue

            py = python_num.iloc[:n].reset_index(drop=True)
            ex = excel_num.iloc[:n].reset_index(drop=True)
            diff = (py - ex).abs()
            max_diff_global = float(np.nanmax(diff.to_numpy()))
            mean_diff_global = float(np.nanmean(diff.to_numpy()))

            candidates.append({
                "sheet": sheet,
                "header_row": int(r),
                "rows_compared": int(n),
                "max_abs_diff": max_diff_global,
                "mean_abs_diff": mean_diff_global,
                "excel_num": ex,
                "diff": diff,
            })

if not candidates:
    raise ValueError(
        "No se encontro ninguna tabla candidata en Excel con los encabezados esperados."
    )

summary = pd.DataFrame([
    {
        "candidate_id": i,
        "sheet": c["sheet"],
        "header_row": c["header_row"],
        "rows_compared": c["rows_compared"],
        "max_abs_diff": c["max_abs_diff"],
        "mean_abs_diff": c["mean_abs_diff"],
    }
    for i, c in enumerate(candidates)
]).sort_values(["max_abs_diff", "mean_abs_diff"]).reset_index(drop=True)

best_id = int(summary.loc[0, "candidate_id"])
best = candidates[best_id]
tol = 1e-6

print("Bloques candidatos detectados en Excel:")
display(summary.drop(columns=["candidate_id"]))

print(f"\nMejor bloque: hoja={best['sheet']}, fila_encabezado={best['header_row']}, filas={best['rows_compared']}")
print(f"Diferencia maxima global: {best['max_abs_diff']:.12f}")
print(f"Coinciden con tolerancia {tol}: {bool((best['diff'] <= tol).all().all())}")

print("\nMaxima diferencia por columna (mejor bloque):")
display(best["diff"].max().sort_values(ascending=False).to_frame("max_abs_diff"))

mismatch_mask = best["diff"] > tol
if mismatch_mask.any().any():
    mismatch_positions = np.argwhere(mismatch_mask.to_numpy())
    rows = []
    py = python_num.iloc[:best['rows_compared']].reset_index(drop=True)
    ex = best["excel_num"]
    for rr, cc in mismatch_positions:
        col = best["diff"].columns[cc]
        rows.append({
            "fila": int(rr),
            "columna": col,
            "python": float(py.iat[rr, cc]),
            "excel": float(ex.iat[rr, cc]),
            "abs_diff": float(best["diff"].iat[rr, cc]),
        })
    print("\nDiferencias detectadas en el mejor bloque:")
    display(pd.DataFrame(rows))
else:
    print("\nNo hay diferencias numericas: Python y Excel coinciden.")

Bloques candidatos detectados en Excel:


,sheet,header_row,rows_compared,max_abs_diff,mean_abs_diff
0,EjercicioPropuesto,43,6,4.440892e-16,5.319819e-17
1,EjercicioPropuesto,33,6,1.236068e+00,5.737195e-01



Mejor bloque: hoja=EjercicioPropuesto, fila_encabezado=43, filas=6
Diferencia maxima global: 0.000000000000
Coinciden con tolerancia 1e-06: True

Maxima diferencia por columna (mejor bloque):


,max_abs_diff
pais_Chile,4.440892e-16
pais_Brasil,4.440892e-16
nSatis_neutral,4.440892e-16
edad,8.326673e-17
sexo_F,0.000000e+00
sexo_M,0.000000e+00
pais_Ecuador,0.000000e+00
pais_España,0.000000e+00
nSatis_meGusta,0.000000e+00
nSatis_noMeGusta,0.000000e+00



No hay diferencias numericas: Python y Excel coinciden.
